In [1]:
# === Run this cell and paste me the full output ===

import trl, transformers
print("trl version:", getattr(trl, "__version__", "unknown"))
print("transformers version:", transformers.__version__)
print()
print("Does trl export SFTConfig?", hasattr(trl, "SFTConfig"))
print("trl top-level names containing 'SFT':", [n for n in dir(trl) if "SFT" in n])
print()

# Look at the RAW data, before any of our sanitizing, to see the true schema
import json
path = "/kaggle/input/datasets/bayazidhs/marin-vibe/marin_tool_dataset.jsonl"
with open(path) as f:
    first_line = f.readline()
row = json.loads(first_line)
print("Top-level keys in one row:", list(row.keys()))
print()
print("First message object, raw:")
print(json.dumps(row["messages"][0], indent=2))
print()
print("All messages in this row, raw:")
print(json.dumps(row["messages"], indent=2))

ModuleNotFoundError: No module named 'trl'

In [2]:
# ===== Kaggle Notebook: Marin Tool Caller SLM Training (Unsloth) =====
# Upload this notebook to Kaggle along with your marin_tool_dataset.jsonl
# Turn on GPU T4 x2 or P100

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U --no-deps "trl>=1.4.0" peft accelerate bitsandbytes
# trl did a MAJOR v1 rewrite in 2026 (0.8.6 -> 1.4.0+ is not a small bump).
# IMPORTANT: if you are re-running this after a previous attempt in the same
# Kaggle session, you MUST restart the kernel now (Run > Restart Session)
# before running this cell. pip installing a new version on disk does NOT
# retroactively update a module that Python already imported earlier in
# this session -- `import trl` will keep returning the OLD cached module
# until the kernel restarts, no matter how many times you pip install -U.
import unsloth
import trl
from packaging.version import Version
print("trl version actually loaded:", trl.__version__)
if Version(trl.__version__) < Version("1.0.0"):
    raise RuntimeError(
        "Still seeing an old trl version in this session even after "
        "installing a newer one. This almost always means the kernel "
        "needs a restart (Run > Restart Session in Kaggle), then re-run "
        "all cells from the top. Do NOT just re-run this cell."
    )

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset

# 1. Configuration
max_seq_length = 2048 # Good enough for tool schemas + short query
dtype = None # Auto detection
load_in_4bit = True # Use 4bit quantization to reduce memory usage

# We recommend Qwen2.5 1.5B or 3B for local CPU/RAM inference (16GB RAM)
# 1.5B is much faster on CPU, 3B is smarter. Let's try 1.5B Instruct first.
model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Format Dataset (ChatML)
# The dataset we generated is already in {"messages": [...]} format.
# Unsloth/Transformers standardizes this via apply_chat_template.

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # We use standard ChatML
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

# --- FIX: sanitize every message so "content" is never None ---
# Root cause of the TypeError: "can only concatenate str (not NoneType) to str"
# is that some messages (very commonly assistant turns that issue a tool call)
# have content=None instead of a string. The ChatML jinja template does
# `message['content'] + ...` and crashes on None. We coerce every message's
# content to a string before it ever reaches apply_chat_template.
import json

def sanitize_messages(messages):
    fixed = []
    for msg in messages:
        content = msg.get("content", None)
        if content is None:
            if msg.get("tool_calls"):
                # Render tool calls as visible text instead of leaving content=None
                content = json.dumps(msg["tool_calls"])
            else:
                content = ""
        elif not isinstance(content, str):
            # Guard against dicts/lists slipping into content
            content = json.dumps(content)
        fixed.append({**msg, "content": content})
    return fixed

def formatting_prompts_func(examples):
    convos = [sanitize_messages(m) for m in examples["messages"]]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/bayazidhs/marin-vibe/marin_tool_dataset.jsonl",
    split="train",
)

# --- Diagnostic: count messages with content=None. Note: 1 per row is ---
# EXPECTED here, since each row's assistant turn uses tool_calls instead of
# content. Only worry if this number is much higher than your row count.
def count_none_content(examples):
    n = 0
    for convo in examples["messages"]:
        for msg in convo:
            if msg.get("content", None) is None:
                n += 1
    return n

sample_n = min(len(dataset), 2000)
none_count = sum(count_none_content(dataset[i:i+1]) for i in range(sample_n))
print(f"[diagnostic] messages with content=None in first {sample_n} rows: {none_count} "
      f"(~1 per row is normal — that's the tool_calls turn)")

dataset = dataset.map(formatting_prompts_func, batched = True)

# --- Compatibility patch -----------------------------------------------
# transformers.Trainer.__init__() no longer accepts `tokenizer=` (renamed
# to `processing_class=`), but trl's SFTTrainer (older versions) still
# forwards `tokenizer=` down via super().__init__(**kwargs). Reaching into
# unsloth's internal closures to fix this is fragile and version-specific.
# Instead we wrap the CLASS ATTRIBUTE `transformers.Trainer.__init__`
# itself (currently unsloth's patched version) with an outer translator.
# `super().__init__(**kwargs)` resolves this attribute dynamically at call
# time, so whatever sits underneath (unsloth's wrapper, then the real HF
# Trainer.__init__) receives `processing_class=` instead of `tokenizer=`,
# regardless of how those inner layers are implemented.
import transformers

if not getattr(transformers.Trainer.__init__, "_tokenizer_compat_patched", False):
    _inner_trainer_init = transformers.Trainer.__init__

    def _tokenizer_compat_trainer_init(self, *args, __inner=_inner_trainer_init, **kwargs):
        # __inner is bound as a default arg at definition time, so this
        # function permanently owns its own reference to the function it
        # wraps -- safe even if this cell is re-run multiple times without
        # a kernel restart (which previously caused infinite recursion,
        # since a plain captured-by-name reference got reassigned out from
        # under an already-wrapped function on the second re-run).
        if "tokenizer" in kwargs:
            tok = kwargs.pop("tokenizer")
            kwargs.setdefault("processing_class", tok)
        return __inner(self, *args, **kwargs)

    _tokenizer_compat_trainer_init._tokenizer_compat_patched = True
    transformers.Trainer.__init__ = _tokenizer_compat_trainer_init
    print("[patch] transformers.Trainer.__init__ wrapped to translate tokenizer= -> processing_class=")
else:
    print("[patch] transformers.Trainer.__init__ already patched in this session; skipping re-wrap")
# -------------------------------------------------------------------------

# 4. Training
# trl's API has shifted across versions (old: TrainingArguments + separate
# SFTTrainer kwargs + tokenizer=; new: everything in SFTConfig +
# processing_class=). Rather than assume one shape, we introspect whatever
# trl actually resolved to at install time and build the right call.
import inspect
from trl import SFTTrainer
try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

common_training_kwargs = dict(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 200, # Set to num_train_epochs = 1 for full run
    # num_train_epochs = 2, # Use this instead of max_steps for full training
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

if SFTConfig is not None and "args" in sft_trainer_params:
    # trl v1 renamed max_seq_length -> max_length, and added `padding_free`
    # (defaults True in some versions), which raises a ValueError when
    # packing=False and max_length isn't explicitly enforced. We detect
    # the right field names/options dynamically rather than hardcoding
    # one version's API.
    sft_config_params = inspect.signature(SFTConfig.__init__).parameters

    seq_len_kwarg = {}
    if "max_length" in sft_config_params:
        seq_len_kwarg["max_length"] = max_seq_length
    elif "max_seq_length" in sft_config_params:
        seq_len_kwarg["max_seq_length"] = max_seq_length

    extra_kwarg = {}
    if "padding_free" in sft_config_params:
        # We're not packing, so disable padding_free to avoid:
        # "When padding_free=True without packing, max_length is not enforced"
        extra_kwarg["padding_free"] = False

    training_args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 2,
        packing = False, # Can make training 5x faster for short sequences
        **seq_len_kwarg,
        **extra_kwarg,
        **common_training_kwargs,
    )
    trainer_kwargs = dict(model = model, train_dataset = dataset, args = training_args)
else:
    # Old trl: plain TrainingArguments + those kwargs go directly on SFTTrainer
    from transformers import TrainingArguments
    training_args = TrainingArguments(**common_training_kwargs)
    trainer_kwargs = dict(
        model = model,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = training_args,
    )

# Current HF Trainer wants `processing_class=`; older ones want `tokenizer=`.
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

print("Building SFTTrainer with kwargs:", list(trainer_kwargs.keys()))
trainer = SFTTrainer(**trainer_kwargs)

trainer_stats = trainer.train()

# 5. Export to GGUF (for local Ollama CPU inference)
print("Exporting model to GGUF format...")
# Save to 4-bit Q4_K_M GGUF (Perfect balance of size and accuracy for 16GB RAM)
model.save_pretrained_gguf("marin_tool_caller", tokenizer, quantization_method = "q4_k_m")

print("Done! Download the .gguf file from the 'marin_tool_caller' folder and use it in Ollama.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_wma5b7qv/tokenizer_config.json.


...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:08<00:00,  8.33s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:12<00:00, 12.68s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/marin_tool_caller`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10043-mix-0ac9dfb (app-b10043-mix-0ac9dfb-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['marin_tool_caller_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversio